In [19]:
import numpy as np

####  le role de ce algorithme est minimiser une fct de perte en utilisant des dériveés ( gradient et hessienne)
class XGBoostMinimal:
    ### Préparation des hyperparamètres pour mon model [constructeur] #####
    def __init__(self, n_estimators=3, learning_rate=0.1, reg_lambda=1):
        self.n_estimators = n_estimators  # Nombre d'arbres
        self.learning_rate = learning_rate # Vitesse d'apprentissage (eta)
        self.reg_lambda = reg_lambda       # Régularisation (empêche l'overfitting)
        self.trees = []                    # Liste pour stocker nos arbres
        self.base_prediction = None        # La prédiction initiale (moyenne)

    def _log_loss_gradient(self, y_true, y_pred):
        # ÉTAPE 1 : Calcul du Gradient (Première dérivée)
        # Pour une classification logistique, c'est (prédiction - réel)
        return y_pred - y_true

    def _log_loss_hessian(self, y_true, y_pred):
        # ÉTAPE 2 : Calcul du Hessien (Deuxième dérivée)
        # Pour la logistique : prédiction * (1 - prédiction)
        return y_pred * (1 - y_pred)

    def fit(self, X, y):
        # Initialisation : On commence par prédire la moyenne (souvent 0.5)
        self.base_prediction = np.mean(y) ## valeur réellé
        current_predictions = np.full(y.shape, self.base_prediction) ## prédiction actuelle

        for i in range(self.n_estimators):
            # ÉTAPE 3 : Calcul des "Résidus" Spéciaux (Gradients et Hessiens)
            ## Résidus = ValeurRéelle - ValeurPrédictionActuelle
            gradients = self._log_loss_gradient(y, current_predictions)
            hessians = self._log_loss_hessian(y, current_predictions)

            # ÉTAPE 4 : Construire un arbre qui prédit comment corriger l'erreur
            tree = self._build_boosting_tree(X, gradients, hessians)### le gain ( donne une valeur numerique exact por faire les calcules suivant )
            self.trees.append(tree)

            # ÉTAPE 5 : Mettre à jour les prédictions
            # prédiction_neuve = prédiction_ancienne + (learning_rate * correction_de_l_arbre)
            update = self._predict_tree(X, tree) ### correction de l'arber
            current_predictions += self.learning_rate * update
            print(f"Arbre {i+1} entraîné.")

    def _calculate_leaf_score(self, G, H):
        # ÉTAPE CLÉ : Le calcul du poids d'une feuille dans XGBoost
        # Formule : - Somme(Gradients) / (Somme(Hessiens) + Lambda)
        return -np.sum(G) / (np.sum(H) + self.reg_lambda)

    def _build_boosting_tree(self, X, G, H):
        # Ici, on simplifie : on crée juste un "Stump" (arbre à 1 seul split)
        # Dans un vrai XGBoost, on ferait une récursion comme pour CART
        best_gain = 0
        best_split = None
        
        # Score de base (Gain)
        def similarity_score(G_sum, H_sum):
            return (G_sum**2) / (H_sum + self.reg_lambda)

        # On cherche le meilleur split binaire
        for feature_idx in range(X.shape[1]):
            thresholds = np.unique(X[:, feature_idx])
            for t in thresholds:
                left_mask = X[:, feature_idx] <= t
                right_mask = ~left_mask
                
                if not any(left_mask) or not any(right_mask): continue
                
                # ÉTAPE 6 : Calcul du Gain de Structure
                gain = (similarity_score(np.sum(G[left_mask]), np.sum(H[left_mask])) +
                        similarity_score(np.sum(G[right_mask]), np.sum(H[right_mask])) -
                        similarity_score(np.sum(G), np.sum(H)))
                ### faire un découpage pour trouver le gain optimal ###
                if gain > best_gain:
                    best_gain = gain
                    best_split = {
                        'feature': feature_idx,
                        'threshold': t,
                        'left_val': self._calculate_leaf_score(G[left_mask], H[left_mask]),
                        'right_val': self._calculate_leaf_score(G[right_mask], H[right_mask])
                    }
        return best_split

    def _predict_tree(self, X, tree):
        # Applique les corrections d'un seul arbre
        if tree is None: return np.zeros(len(X))
        return np.where(X[:, tree['feature']] <= tree['threshold'], tree['left_val'], tree['right_val'])

    def predict_proba(self, X):
        # Somme des prédictions de tous les arbres
        val = np.full(len(X), self.base_prediction)
        for tree in self.trees:
            val += self.learning_rate * self._predict_tree(X, tree)
        # Transformation Sigmoïde pour avoir une probabilité entre 0 et 1
        return 1 / (1 + np.exp(-val))

# --- TEST ---
import numpy as np

# 1. Tes données brutes (exemple simplifié)
# Outlook: Sunny(0), Overcast(1), Rain(2)
#Temp: Hot(0), Mild(1), Cool(2)
# Humidity: High(0), Normal(1)
# Wind: Weak(0), Strong(1)
X_tennis = np.array([
    [0, 0, 0, 0], # Sunny, Hot, High, Weak
    [0, 0, 0, 1], # Sunny, Hot, High, Strong
    [1, 0, 0, 0], # Overcast, Hot, High, Weak
    [2, 1, 0, 0], # Rain, Mild, High , Weak
    [2, 2, 1, 0], #Rain, Cool, Normal, Weak
    [2, 2, 1, 1], #Rain, Cool, Normal, Strong
    [1, 2, 1, 1], #Overcast, Cool, Normal, Strong 
    [0, 1, 0, 0], #Sunny, Mild, High, Weak 
    [0, 2, 1, 0], #Sunny, Cool, Normal, Weak
    [2, 1, 1, 0], #Rain, Mild, Normal, Weak
    [0, 1, 1, 1], #Sunny, Mild, Normal, Strong
    [1, 1, 0, 1], #Overcast, Mild, High, Strong
    [1, 0, 1, 0], #Overcast, Hoy, Normal, Weak
    [2, 1, 0, 1] #Rain, Mild, High, Strong
])

# Target: No(0), Yes(1)
y_tennis = np.array([0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0]) # (exemple)

# 2. Appliquer ton modèle XGBoost From Scratch
model = XGBoostMinimal(n_estimators=3, learning_rate=0.1)
model.fit(X_tennis, y_tennis)

# 3. Prédire pour un nouveau jour (ex: Sunny, Hot, Normal, Weak)
nouveau_jour = np.array([[0, 0, 1, 0]])
probabilite = model.predict_proba(nouveau_jour)

print(f"Probabilité de jouer : {probabilite[0] * 100:.2f}%")

#X_test = np.array([[1, 2], [2, 3], [3, 1], [4, 3]])
#y_test = np.array([0, 0, 1, 1])
#model = XGBoostMinimal(n_estimators=5)
#model.fit(X_test, y_test)
#print("Probabilités prédites :", model.predict_proba(X_test))

Arbre 1 entraîné.
Arbre 2 entraîné.
Arbre 3 entraîné.
Probabilité de jouer : 66.56%


In [20]:
############################### ----- Algorithme Naive de Bayes -----------################

import numpy as np

class NaiveBayesTennis:
    def fit(self, X, y):
        self.X = X
        self.y = y
        self.classes = np.unique(y)
        self.parameters = {}

        for c in self.classes:
            X_c = X[y == c]
            self.parameters[c] = {
                "prior": len(X_c) / len(y),
                "counts": []
            }
            for col in range(X.shape[1]):
                val_counts = {}
                unique_vals = np.unique(X[:, col])
                for v in unique_vals:
                    count = (X_c[:, col] == v).sum()
                    # Lissage de Laplace pour éviter les probabilités nulles
                    val_counts[v] = (count + 1) / (len(X_c) + len(unique_vals))
                self.parameters[c]["counts"].append(val_counts)

    def predict(self, X_test):
        predictions = []
        for sample in X_test:
            probs = {}
            for c in self.classes:
                prob = self.parameters[c]["prior"]
                for col, val in enumerate(sample):
                    prob *= self.parameters[c]["counts"][col].get(val, 1e-6)
                probs[c] = prob
            predictions.append(max(probs, key=probs.get))
        return np.array(predictions)

# --- DATASET TENNIS ---
X = np.array([
    [0,0,0,0], [0,0,0,1], [1,0,0,0], [2,1,0,0], [2,2,1,0],
    [2,2,1,1], [1,2,1,1], [0,1,0,0], [0,2,1,0], [2,1,1,0],
    [0,1,1,1], [1,1,0,1], [1,0,1,0], [2,1,0,1]
])
y = np.array([0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0]) # 0=No, 1=Yes

# Entraînement
nb = NaiveBayesTennis()
nb.fit(X, y)

# ======================================================
# PARTIE TEST : NOUVELLE JOURNÉE
# ======================================================

# Imaginons un jour avec : 
# Outlook: Rain (2), Temp: Cool (2), Humidity: Normal (1), Wind: Strong (1)
nouveau_jour = np.array([[2, 2, 1, 1]])

# Dictionnaire pour traduire les chiffres en mots (pour l'affichage)
desc = {
    "Outlook": {0: "Sunny", 1: "Overcast", 2: "Rain"},
    "Temp": {0: "Hot", 1: "Mild", 2: "Cool"},
    "Hum": {0: "High", 1: "Normal"},
    "Wind": {0: "Weak", 1: "Strong"}
}

# 1. Faire la prédiction
prediction = nb.predict(nouveau_jour)
resultat = "OUI, on peut jouer !" if prediction[0] == 1 else "NON, trop risqué."

# 2. Affichage propre du test
print("--- TEST DE PRÉDICTION ---")
print(f"Conditions météo :")
print(f"  - Ciel      : {desc['Outlook'][nouveau_jour[0][0]]}")
print(f"  - Temp      : {desc['Temp'][nouveau_jour[0][1]]}")
print(f"  - Humidité  : {desc['Hum'][nouveau_jour[0][2]]}")
print(f"  - Vent      : {desc['Wind'][nouveau_jour[0][3]]}")
print("-" * 25)
print(f"RÉSULTAT : {resultat}")
print("-" * 25)

--- TEST DE PRÉDICTION ---
Conditions météo :
  - Ciel      : Rain
  - Temp      : Cool
  - Humidité  : Normal
  - Vent      : Strong
-------------------------
RÉSULTAT : OUI, on peut jouer !
-------------------------
